# 시설 대조와 연결 규칙 검증 — 2026-10-01

승인된 출처의 활성 미연결 시설을 읽기 전용으로 대조한다. 기록 단위는 `sourceId + externalId`이며 주변 후보는 같은 시설로 승인된 장소가 아니다. 이 노트북은 수집·발행·재연결을 수행하지 않는다. 저장소의 Node 의존성, PostgreSQL/PostGIS와 실제 발행 자료가 필요하다. `.env` 값을 출력하지 않는다. 재실행 결과는 DB 상태에 따라 달라질 수 있다.


In [1]:
import json, subprocess, tempfile
from pathlib import Path
from collections import Counter
repo=Path.cwd().resolve()
while not (repo/'scripts/tourism/compare.ts').exists():
    if repo.parent==repo: raise RuntimeError('Book해도 저장소 안에서 실행하세요')
    repo=repo.parent
with tempfile.TemporaryDirectory() as directory:
    output=Path(directory)/'comparison.json'
    subprocess.run(['npm','run','tourism:compare','--','--source','furano-places','--date','2026-10-01','--output',str(output)],cwd=repo,capture_output=True,text=True,check=True)
    comparison=json.loads(output.read_text())
print(json.dumps({'checkedAt':comparison['checkedAt'],'summary':comparison['summary']},ensure_ascii=False,indent=2))


{
  "checkedAt": "2026-10-01T07:57:41.755Z",
  "summary": {
    "NAME_MISMATCH": 10,
    "NO_CATALOG_PLACE_WITHIN_250M": 6
  }
}


In [2]:
rows=comparison['rows']
assert len({(row['sourceId'],row['externalId']) for row in rows})==len(rows)
assert all(row['decision']=='PENDING' and row['approvedPlaceId'] is None for row in rows)
assert dict(Counter(row['reason'] for row in rows))==comparison['summary']
assert all(candidate['regionId']==row['regionId'] and 0<=candidate['distanceMeters']<=comparison['matchingRadiusMeters'] for row in rows for candidate in row['candidates'])
print(json.dumps({'pending':len(rows),'candidates':sum(len(row['candidates']) for row in rows)},indent=2))
print(comparison['matchingSql'])
# SQL parameters: each facility's region, longitude, latitude, and shared radius (250m).


{
  "pending": 16,
  "candidates": 56
}
SELECT id,name_ja AS name,region_id AS "regionId",category,address,website,
ST_Distance(location,ST_SetSRID(ST_MakePoint($2,$3),4326)::geography) AS "distanceMeters"
FROM geo_data.place WHERE region_id=$1 AND ST_DWithin(location,
ST_SetSRID(ST_MakePoint($2,$3),4326)::geography,$4) ORDER BY "distanceMeters",id


In [3]:
review=json.loads((repo/'docs/research/tourism-facility-review-20261001.json').read_text())
index={(row['sourceId'],row['externalId']):row for row in rows}
for note in review['rows']:
    row=index[(note['sourceId'],note['externalId'])]
    assert note['snapshotId']==row['snapshotId'] and note['contentSha256']==row['contentSha256']
    assert note['decision']=='PENDING' and note['approvedPlaceId'] is None
    if note['proposedPlaceId']:
        assert note['proposedPlaceId'] in {candidate['id'] for candidate in row['candidates']}
print(json.dumps(dict(Counter(note['recommendation'] for note in review['rows'])),indent=2))


{
  "NEEDS_IDENTITY_EVIDENCE": 8,
  "ALIAS_CANDIDATE": 1,
  "KEEP_SEPARATE": 1
}


공식 근거: [와인하우스](https://www.furano.ne.jp/winehouse/)는 CSV와 같은 시설명, catalog와 같은 주소를 표시한다. [마르셰 FAQ](https://marche.furano.jp/?page_id=1340)는 1과 2가 도로를 사이에 둔다고 설명한다. 승인된 매핑이 아닌 수동 대조 메모의 근거이며, 나머지 8건에는 독립적인 동일 시설 확인이 없다.

시각 측정은 별도 `tourism-langgraph-review-5-20261001.json`에 저장했다. 6개 요청의 loopback HTTP fixture 결과이며 운영 처리량 측정이 아니다.
